# White Noise (SNR) Robustness Experiment (Colab Runner)

Αυτό το notebook είναι ρυθμισμένο για να τρέχει το πείραμα αξιολόγησης ευρωστίας με λευκό θόρυβο `run_whitenoise_snr.py` κατευθείαν μέσα από το Google Drive σου, εκμεταλλευόμενο τους πόρους του Google Colab.

**SNR Levels**: 40, 30, 20, 10, 5, 0, -5, -10, -20 dB

**Μοντέλα**: IForest, PCA, LOF, MatrixProfile

### 1. Σύνδεση με το Google Drive
Τρέξε το παρακάτω κελί για να δώσεις πρόσβαση στο Colab να διαβάσει και να γράψει στο Google Drive σου.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Μετάβαση στο φάκελο του Project
Εδώ ορίζουμε που ακριβώς βρίσκεται ο φάκελος της διπλωματικής σου μέσα στο Drive.
**ΠΡΟΣΟΧΗ**: Αν έχεις αλλάξει το όνομα του φακέλου ή τον έχεις βάλει μέσα σε άλλο φάκελο, άλλαξε το path παρακάτω:

In [ ]:
import os

# Το σωστό path για συγχρονισμένους φακέλους από υπολογιστή
PROJECT_PATH = '/content/drive/Othercomputers/Ο φορητός υπολογιστής μου/thesis_timeseries'

os.chdir(PROJECT_PATH)
print("Current working directory:", os.getcwd())

# Έλεγχος ότι βρισκόμαστε στο σωστό μέρος
if os.path.exists('src/experiments/corruption_tsbuad/run_whitenoise_snr.py'):
    print("✅ Το Path είναι σωστό! Το αρχείο experiment βρέθηκε.")
else:
    print("❌ Το αρχείο experiment ΔΕΝ βρέθηκε. Έλεγξε το PROJECT_PATH.")

### 3. Εγκατάσταση Εξαρτήσεων (Dependencies)
Επειδή το Colab ξεκινάει με ένα καθαρό περιβάλλον Python, πρέπει να εγκαταστήσουμε ό,τι λείπει (π.χ. το TSB-UAD).

In [ ]:
# Εγκατάσταση βασικών dependencies
!pip install stumpy tqdm scikit-learn scipy statsmodels --quiet

# Εγκατάσταση του TSB-UAD αν δεν είναι ήδη εγκατεστημένο
import sys
if not os.path.exists('TSB-UAD/TSB_UAD/__init__.py'):
    print("TSB-UAD module needed, applying hotfix for Colab...")

# Προσθέτουμε τα paths στο python path
for p in ['TSB-UAD', 'src', '.']:
    full = os.path.join(os.getcwd(), p)
    if full not in sys.path:
        sys.path.insert(0, full)

print("✅ Dependencies ready.")

### 4. Έλεγχος Δεδομένων
Βεβαιωνόμαστε ότι το subset CSV και τα δεδομένα υπάρχουν.

In [ ]:
import pandas as pd

subset_csv = 'results/tables/robust_subset_TSB.csv'
if os.path.exists(subset_csv):
    df_sub = pd.read_csv(subset_csv)
    print(f"✅ Subset CSV found: {len(df_sub)} datasets")
    
    # Έλεγχος πόσα αρχεία υπάρχουν στο disk
    existing = sum(1 for fp in df_sub['filepath'] if os.path.exists(fp))
    print(f"   Αρχεία που βρέθηκαν στο disk: {existing}/{len(df_sub)}")
    
    if existing < len(df_sub):
        print(f"   ⚠️  Λείπουν {len(df_sub) - existing} αρχεία!")
        missing_example = [fp for fp in df_sub['filepath'] if not os.path.exists(fp)][:3]
        print(f"   Παράδειγμα paths που λείπουν: {missing_example}")
else:
    print(f"❌ Subset CSV NOT found at: {subset_csv}")

### 5. Εκτέλεση του Πειράματος 🎉

Τρέχουμε το κεντρικό script. Παράμετροι:
- `--workers 2`: Τα δωρεάν Colab instances έχουν συνήθως μόνο 2 cores
- `--models`: Ποια μοντέλα να αξιολογηθούν (IForest, PCA, MP, LOF)

Το checkpointing δουλεύει κανονικά! Αν κλείσει το Colab, την επόμενη φορά θα συνεχίσει από εκεί που έμεινε.

In [ ]:
# Τρέξε ΟΛΑ τα μοντέλα (IForest, PCA, LOF, MP)
!python src/experiments/corruption_tsbuad/run_whitenoise_snr.py \
    --models IForest PCA LOF MP \
    --workers 2

### 5b. (Εναλλακτικά) Τρέξε μοντέλα ένα-ένα
Αν ο χρόνος είναι περιορισμένος, μπορείς να τρέξεις πρώτα τα γρήγορα μοντέλα (IForest, PCA, LOF) και μετά το αργό (MP).

In [ ]:
# Γρήγορα μοντέλα πρώτα
!python src/experiments/corruption_tsbuad/run_whitenoise_snr.py --models IForest PCA LOF --workers 2

In [ ]:
# Αργό μοντέλο (MatrixProfile) - μπορεί να θέλει αρκετή ώρα
!python src/experiments/corruption_tsbuad/run_whitenoise_snr.py --models MP --workers 2

### 6. Smoke Test (Προαιρετικό)
Δοκιμαστική εκτέλεση με 3 datasets και 2 SNR levels για να βεβαιωθείς ότι όλα δουλεύουν.

In [ ]:
!python src/experiments/corruption_tsbuad/run_whitenoise_snr.py --test --models IForest --workers 2

### 7. Έλεγχος Αποτελεσμάτων
Μετά την ολοκλήρωση, ελέγχουμε τα αποτελέσματα.

In [ ]:
import pandas as pd

results_dir = 'results/experiments/white_noise_snr'

# Checkpoint (raw results)
checkpoint_path = os.path.join(results_dir, 'checkpoint.csv')
if os.path.exists(checkpoint_path):
    df_cp = pd.read_csv(checkpoint_path)
    print(f"📊 Checkpoint: {len(df_cp)} results")
    print(f"   Models: {df_cp['model'].unique().tolist()}")
    print(f"   SNR levels: {sorted(df_cp['snr_db'].unique().tolist())}")
    print(f"   Datasets: {df_cp['file'].nunique()}")
else:
    print("❌ No checkpoint found yet.")

# Summary
summary_path = os.path.join(results_dir, 'summary.csv')
if os.path.exists(summary_path):
    df_sum = pd.read_csv(summary_path)
    print(f"\n📈 Summary: {len(df_sum)} rows")
    print(df_sum.head(10).to_string(index=False))
else:
    print("\n⏳ Summary not yet generated (experiment still running?).")

### 8. Quick Preview Plot
Γρήγορο preview των αποτελεσμάτων (AUC-ROC vs SNR).

In [ ]:
import matplotlib.pyplot as plt

summary_path = 'results/experiments/white_noise_snr/summary.csv'
if os.path.exists(summary_path):
    df_sum = pd.read_csv(summary_path)
    
    MODEL_STYLES = {
        'IForest': {'color': '#D62728', 'marker': 's'},
        'PCA':     {'color': '#1F77B4', 'marker': 'o'},
        'LOF':     {'color': '#2CA02C', 'marker': '^'},
        'MP':      {'color': '#FF7F0E', 'marker': 'D'},
    }
    
    plt.style.use('classic')
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for model in df_sum['model'].unique():
        m_df = df_sum[df_sum['model'] == model].sort_values('snr_db')
        style = MODEL_STYLES.get(model, {'color': 'gray', 'marker': 'x'})
        ax.plot(m_df['snr_db'], m_df['AUC_ROC_mean'], 
                label=model, marker=style['marker'], color=style['color'],
                linewidth=2, markersize=8)
    
    ax.set_xlabel('SNR (dB)', fontsize=12)
    ax.set_ylabel('AUC-ROC (mean)', fontsize=12)
    ax.set_title('Model Robustness: AUC-ROC vs White Noise SNR', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.invert_xaxis()
    plt.tight_layout()
    plt.show()
else:
    print("⏳ Summary not available yet.")